# 01 — Dataset overview and sanity checks

**Goal.** Load the Czech picture-description transcripts, confirm the data is what we think
it is, and fix the split convention that every later notebook follows.

**Data.** `fileDataset/` from the project Drive folder. Each participant described the same
hand-drawn lakeshore scene for 60 seconds (Task 04 of the DigiDiaDem battery). The text is an
automatic transcription; no audio is available.

**Split convention used throughout these notebooks:**

| folder | name here | labels | what we use it for |
|---|---|---|---|
| `overview/` | `train.extra` | none | LLM feature *discovery* and prompt experiments |
| `train/negative`, `train/positive` | `train` | 0 / 1 | all modelling, all model selection |
| `test/negative`, `test/positive` | `test` | 0 / 1 | **not used in these notebooks at all** |

Positive = original diagnosis code 2 (mild cognitive impairment) or 3 (mild dementia).
Negative = code 0 (cognitively normal). Code 1 is excluded upstream.

> **Rule for this consultation:** the labelled test set is never touched — not for feature
> design, not for selection, not for reporting. Notebooks 03–06 load `train/` only. We keep a
> count of the test files here purely so we can describe the dataset honestly.

In [6]:
import re
import zipfile
from pathlib import Path

import pandas as pd

DATA = Path("fileDataset")

# The Drive folder ships a zip; unpack it once if the folder is not there yet.
if not DATA.exists() and Path("fileDataset.zip").exists():
    with zipfile.ZipFile("fileDataset.zip") as zf:
        zf.extractall(".")
    print("unpacked fileDataset.zip")

# Some zips nest one extra level.
if not (DATA / "train").exists() and (DATA / "fileDataset" / "train").exists():
    DATA = DATA / "fileDataset"

print("data root:", DATA.resolve())
print("subfolders:", sorted(p.name for p in DATA.iterdir() if p.is_dir()))

data root: /content/fileDataset


FileNotFoundError: [Errno 2] No such file or directory: 'fileDataset'

## Load every transcript into one dataframe

One row per participant. `split` and `label` come from the folder the file sits in — the
directory layout *is* the ground truth, so we read it rather than re-deriving it.

In [ ]:
FOLDERS = [
    ("overview",       "train.extra", None),
    ("train/negative", "train",       0),
    ("train/positive", "train",       1),
    ("test/negative",  "test",        0),
    ("test/positive",  "test",        1),
]

rows = []
for folder, split, label in FOLDERS:
    for path in sorted((DATA / folder).glob("*.txt")):
        rows.append(
            {
                "file": path.name,
                "split": split,
                "label": label,
                "text": path.read_text(encoding="utf-8").strip(),
            }
        )

df = pd.DataFrame(rows)
print(f"loaded {len(df)} transcripts")
df.head(3)

## How many documents are in each split?

This is the first thing to show the supervisor: the working sample is 241 labelled documents,
of which 70 are positive. Everything downstream is constrained by those numbers.

In [ ]:
counts = (
    df.groupby(["split", "label"], dropna=False)
    .size()
    .rename("n")
    .reset_index()
)
counts["label"] = counts["label"].map({0: "negative", 1: "positive"}).fillna("unlabelled")
print(counts.to_string(index=False))

train = df[df.split == "train"]
print(f"\ntrain: {len(train)} docs, {int(train.label.sum())} positive "
      f"({train.label.mean():.1%})")
print(f"majority-class accuracy on train = {max(train.label.mean(), 1 - train.label.mean()):.3f}")

The class balance matters more than it looks. Roughly 29% of the training documents are
positive, so a model that always predicts "negative" already scores 71% accuracy. **Accuracy
alone will therefore never tell us whether a model has learned anything** — this is why the
later notebooks report balanced accuracy, macro-F1 and AUC alongside it.

## Sanity check 1 — duplicates

If the same transcript appeared in two splits, any evaluation would be contaminated.

In [ ]:
dupes = df.text.duplicated().sum()
print(f"duplicate transcripts: {dupes}")
print(f"empty transcripts    : {(df.text.str.len() == 0).sum()}")
assert dupes == 0, "duplicate transcripts found — investigate before modelling"
assert (df.text.str.len() > 0).all()
print("\nOK — every transcript is unique and non-empty.")

## Sanity check 2 — how long are these texts?

Transcript length turns out to matter a great deal later, so we measure it now.

In [ ]:
WORD = re.compile(r"\w+", re.UNICODE)

df["n_word"] = df.text.str.split().str.len()
df["n_char"] = df.text.str.len()
df["n_type"] = df.text.str.lower().apply(lambda t: len(set(WORD.findall(t))))
df["ttr"] = df.n_type / df.n_word                       # type-token ratio: vocabulary richness
df["n_sent"] = df.text.apply(lambda t: max(1, len(re.findall(r"[.!?]+", t))))
df["mlu"] = df.n_word / df.n_sent                       # mean length of utterance
df["n_comma"] = df.text.str.count(",")

SURFACE = ["n_word", "n_type", "ttr", "n_sent", "mlu", "n_comma", "n_char"]

# refresh the training view now that the surface columns exist
train = df[df.split == "train"]

print(df.groupby("split").n_word.describe()[["count", "mean", "50%", "min", "max"]].round(1).to_string())

The transcripts are short — a median of roughly 65 words. This is a real constraint on the
whole project: there is a limit to how much "discourse structure" an LLM can measure in
sixty words, and it makes some of the constructs in the thesis proposal harder to
operationalise than they sound.

## Sanity check 3 — do the two classes differ on simple surface measures?

Before involving any LLM, we check whether trivially computable numbers already separate the
groups. If they do, every later feature has to beat them to be worth anything.

In [ ]:
from scipy.stats import mannwhitneyu

print(f"{'measure':10}{'negative':>12}{'positive':>12}{'Mann-Whitney p':>18}")
for col in SURFACE:
    a = train.loc[train.label == 1, col]
    b = train.loc[train.label == 0, col]
    p = mannwhitneyu(a, b).pvalue
    print(f"{col:10}{b.median():12.2f}{a.median():12.2f}{p:18.4f}")

Participants in the positive group say noticeably less. That is a genuine clinical
observation, but it is also a warning: **any feature that grows with transcript length will
look predictive without measuring anything interesting.** Notebook 03 tests each LLM feature
for exactly this.

## What the transcripts actually look like

Worth reading a couple aloud before trusting any number computed from them.

In [ ]:
for lab, name in [(0, "NEGATIVE (cognitively normal)"), (1, "POSITIVE (MCI / mild dementia)")]:
    ex = train[train.label == lab].iloc[0]
    print(f"--- {name} — {ex.file}, {ex.n_word} words ---")
    print(ex.text)
    print()

## Save the corpus for the other notebooks

In [ ]:
Path("outputs").mkdir(exist_ok=True)
df.to_csv("outputs/corpus.csv", index=False)
print(f"wrote outputs/corpus.csv  ({len(df)} rows, {len(df.columns)} columns)")

### Summary

* 388 transcripts, all unique: 241 labelled training, 61 labelled test (untouched), 86 unlabelled.
* 29% of training documents are positive, so the majority baseline is 0.71 accuracy.
* Median length ~65 words; the positive group produces significantly shorter transcripts.
* **Open data question for the supervisor:** the filenames are sequential counters and carry
  no participant ID, so these transcripts cannot be linked back to the pickle for age, sex or
  education. Controlling for those confounds is impossible until we have that key.